In [0]:
bronze_drivers = spark.table(
    "workspace.transportation_analytics.bronze_drivers"
)

print("Bronze drivers table loaded successfully")
print("Row count:", bronze_drivers.count())

print("\n===== BRONZE DRIVERS SCHEMA =====")
bronze_drivers.printSchema()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable

print("Silver layer libraries loaded successfully")

In [0]:
silver_drivers = bronze_drivers

# Remove extra spaces from text columns
for column_name, data_type in bronze_drivers.dtypes:
    if data_type == "string":
        silver_drivers = silver_drivers.withColumn(
            column_name,
            F.trim(F.col(column_name))
        )

# Remove duplicate driver records
silver_drivers = silver_drivers.dropDuplicates(["driver_id"])

print("Drivers data cleaned successfully")
print("Row count:", silver_drivers.count())

print("\n===== SILVER DRIVERS SCHEMA =====")
silver_drivers.printSchema()

In [0]:
silver_table = "workspace.transportation_analytics.silver_drivers"

if not spark.catalog.tableExists(silver_table):
    silver_drivers.write.format("delta").saveAsTable(silver_table)
    print("Silver drivers table created")
else:
    (
        DeltaTable.forName(spark, silver_table)
        .alias("target")
        .merge(
            silver_drivers.alias("source"),
            "target.driver_id = source.driver_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print("Silver drivers table updated using MERGE")

silver_drivers = spark.table(silver_table)

print("Row count:", silver_drivers.count())

print("\n===== SILVER DRIVERS SCHEMA =====")
silver_drivers.printSchema()

In [0]:
bronze_customers = spark.table(
    "workspace.transportation_analytics.bronze_customers"
)

print("Bronze customers table loaded successfully")
print("Row count:", bronze_customers.count())

print("\n===== BRONZE CUSTOMERS SCHEMA =====")
bronze_customers.printSchema()

In [0]:
silver_customers = bronze_customers

# Remove extra spaces from text columns
for column_name, data_type in bronze_customers.dtypes:
    if data_type == "string":
        silver_customers = silver_customers.withColumn(
            column_name,
            F.trim(F.col(column_name))
        )

# Remove duplicate customer records
silver_customers = silver_customers.dropDuplicates(["customer_id"])

print("Customers data cleaned successfully")
print("Row count:", silver_customers.count())

print("\n===== SILVER CUSTOMERS SCHEMA =====")
silver_customers.printSchema()

In [0]:
silver_table = "workspace.transportation_analytics.silver_customers"

if not spark.catalog.tableExists(silver_table):
    silver_customers.write.format("delta").saveAsTable(silver_table)
    print("Silver customers table created")
else:
    (
        DeltaTable.forName(spark, silver_table)
        .alias("target")
        .merge(
            silver_customers.alias("source"),
            "target.customer_id = source.customer_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print("Silver customers table updated using MERGE")

silver_customers = spark.table(silver_table)

print("Row count:", silver_customers.count())

print("\n===== SILVER CUSTOMERS SCHEMA =====")
silver_customers.printSchema()

In [0]:
bronze_delivery_events = spark.table(
    "workspace.transportation_analytics.bronze_delivery_events"
)

print("Bronze delivery events table loaded successfully")
print("Row count:", bronze_delivery_events.count())

print("\n===== BRONZE DELIVERY EVENTS SCHEMA =====")
bronze_delivery_events.printSchema()

In [0]:
silver_delivery_events = bronze_delivery_events

# Remove extra spaces from text columns
for column_name, data_type in bronze_delivery_events.dtypes:
    if data_type == "string":
        silver_delivery_events = silver_delivery_events.withColumn(
            column_name,
            F.trim(F.col(column_name))
        )

# Remove duplicate delivery events
silver_delivery_events = silver_delivery_events.dropDuplicates(["event_id"])

print("Delivery events data cleaned successfully")
print("Row count:", silver_delivery_events.count())

print("\n===== SILVER DELIVERY EVENTS SCHEMA =====")
silver_delivery_events.printSchema()

In [0]:
silver_table = "workspace.transportation_analytics.silver_delivery_events"

if not spark.catalog.tableExists(silver_table):
    silver_delivery_events.write.format("delta").saveAsTable(silver_table)
    print("Silver delivery events table created")
    silver_delivery_events.printSchema()
else:
    DeltaTable.forName(spark, silver_table).alias("t") \
        .merge(
            silver_delivery_events.alias("s"),
            "t.event_id = s.event_id"
        ) \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()
    print("Silver delivery events table updated using MERGE")

print("Rows:", spark.table(silver_table).count())

In [0]:
bronze_driver_monthly = spark.table(
    "workspace.transportation_analytics.bronze_driver_monthly_metrics"
)

print("Driver monthly metrics loaded")
print("Rows:", bronze_driver_monthly.count())

In [0]:
silver_driver_monthly = bronze_driver_monthly

for column_name, data_type in bronze_driver_monthly.dtypes:
    if data_type == "string":
        silver_driver_monthly = silver_driver_monthly.withColumn(
            column_name, F.trim(F.col(column_name))
        )

silver_driver_monthly = silver_driver_monthly.dropDuplicates(
    ["driver_id", "month"]
)

print("Driver monthly metrics cleaned")
print("Rows:", silver_driver_monthly.count())

In [0]:
silver_table = "workspace.transportation_analytics.silver_driver_monthly_metrics"

if not spark.catalog.tableExists(silver_table):
    silver_driver_monthly.write.format("delta").saveAsTable(silver_table)
    print("Silver driver monthly metrics table created")
    silver_driver_monthly.printSchema()
else:
    DeltaTable.forName(spark, silver_table).alias("t") \
        .merge(
            silver_driver_monthly.alias("s"),
            "t.driver_id = s.driver_id AND t.month = s.month"
        ) \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()
    print("Silver driver monthly metrics table updated using MERGE")

print("Rows:", spark.table(silver_table).count())

In [0]:
bronze_facilities = spark.table(
    "workspace.transportation_analytics.bronze_facilities"
)

print("Facilities loaded")
print("Rows:", bronze_facilities.count())

In [0]:
silver_facilities = bronze_facilities

for column_name, data_type in bronze_facilities.dtypes:
    if data_type == "string":
        silver_facilities = silver_facilities.withColumn(
            column_name, F.trim(F.col(column_name))
        )

silver_facilities = silver_facilities.dropDuplicates(["facility_id"])

print("Facilities cleaned")
print("Rows:", silver_facilities.count())

In [0]:
silver_table = "workspace.transportation_analytics.silver_facilities"

if not spark.catalog.tableExists(silver_table):
    silver_facilities.write.format("delta").saveAsTable(silver_table)
    print("Silver facilities table created")
    silver_facilities.printSchema()
else:
    DeltaTable.forName(spark, silver_table).alias("t") \
        .merge(
            silver_facilities.alias("s"),
            "t.facility_id = s.facility_id"
        ) \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()
    print("Silver facilities table updated using MERGE")

print("Rows:", spark.table(silver_table).count())

In [0]:
bronze_fuel_purchases = spark.table(
    "workspace.transportation_analytics.bronze_fuel_purchases"
)

print("Fuel purchases loaded")
print("Rows:", bronze_fuel_purchases.count())

In [0]:
silver_fuel_purchases = bronze_fuel_purchases

for column_name, data_type in bronze_fuel_purchases.dtypes:
    if data_type == "string":
        silver_fuel_purchases = silver_fuel_purchases.withColumn(
            column_name, F.trim(F.col(column_name))
        )

silver_fuel_purchases = silver_fuel_purchases.dropDuplicates(
    ["fuel_purchase_id"]
)

print("Fuel purchases cleaned")
print("Rows:", silver_fuel_purchases.count())

In [0]:
silver_table = "workspace.transportation_analytics.silver_fuel_purchases"

if not spark.catalog.tableExists(silver_table):
    silver_fuel_purchases.write.format("delta").saveAsTable(silver_table)
    print("Silver fuel purchases table created")
    silver_fuel_purchases.printSchema()
else:
    DeltaTable.forName(spark, silver_table).alias("t") \
        .merge(
            silver_fuel_purchases.alias("s"),
            "t.fuel_purchase_id = s.fuel_purchase_id"
        ) \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()
    print("Silver fuel purchases table updated using MERGE")

print("Rows:", spark.table(silver_table).count())

In [0]:
bronze_loads = spark.table(
    "workspace.transportation_analytics.bronze_loads"
)

print("Loads loaded")
print("Rows:", bronze_loads.count())

In [0]:
silver_loads = bronze_loads

for column_name, data_type in bronze_loads.dtypes:
    if data_type == "string":
        silver_loads = silver_loads.withColumn(
            column_name, F.trim(F.col(column_name))
        )

silver_loads = silver_loads.dropDuplicates(["load_id"])

print("Loads cleaned")
print("Rows:", silver_loads.count())

In [0]:
silver_table = "workspace.transportation_analytics.silver_loads"

if not spark.catalog.tableExists(silver_table):
    silver_loads.write.format("delta").saveAsTable(silver_table)
    print("Silver loads table created")
    silver_loads.printSchema()
else:
    DeltaTable.forName(spark, silver_table).alias("t") \
        .merge(
            silver_loads.alias("s"),
            "t.load_id = s.load_id"
        ) \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()
    print("Silver loads table updated using MERGE")

print("Rows:", spark.table(silver_table).count())

In [0]:
bronze_maintenance = spark.table(
    "workspace.transportation_analytics.bronze_maintenance_records"
)

print("Maintenance records loaded")
print("Rows:", bronze_maintenance.count())

In [0]:
silver_maintenance = bronze_maintenance

for column_name, data_type in bronze_maintenance.dtypes:
    if data_type == "string":
        silver_maintenance = silver_maintenance.withColumn(
            column_name, F.trim(F.col(column_name))
        )

silver_maintenance = silver_maintenance.dropDuplicates(["maintenance_id"])

print("Maintenance records cleaned")
print("Rows:", silver_maintenance.count())

In [0]:
silver_table = "workspace.transportation_analytics.silver_maintenance_records"

if not spark.catalog.tableExists(silver_table):
    silver_maintenance.write.format("delta").saveAsTable(silver_table)
    print("Silver maintenance records table created")
    silver_maintenance.printSchema()
else:
    DeltaTable.forName(spark, silver_table).alias("t") \
        .merge(
            silver_maintenance.alias("s"),
            "t.maintenance_id = s.maintenance_id"
        ) \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()
    print("Silver maintenance records table updated using MERGE")

print("Rows:", spark.table(silver_table).count())

In [0]:
bronze_routes = spark.table(
    "workspace.transportation_analytics.bronze_routes"
)

print("Routes loaded")
print("Rows:", bronze_routes.count())

In [0]:
silver_routes = bronze_routes

for column_name, data_type in bronze_routes.dtypes:
    if data_type == "string":
        silver_routes = silver_routes.withColumn(
            column_name, F.trim(F.col(column_name))
        )

silver_routes = silver_routes.dropDuplicates(["route_id"])

print("Routes cleaned")
print("Rows:", silver_routes.count())

In [0]:
silver_table = "workspace.transportation_analytics.silver_routes"

if not spark.catalog.tableExists(silver_table):
    silver_routes.write.format("delta").saveAsTable(silver_table)
    print("Silver routes table created")
    silver_routes.printSchema()
else:
    DeltaTable.forName(spark, silver_table).alias("t") \
        .merge(
            silver_routes.alias("s"),
            "t.route_id = s.route_id"
        ) \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()
    print("Silver routes table updated using MERGE")

print("Rows:", spark.table(silver_table).count())

In [0]:
bronze_safety = spark.table(
    "workspace.transportation_analytics.bronze_safety_incidents"
)

print("Safety incidents loaded")
print("Rows:", bronze_safety.count())

In [0]:
silver_safety = bronze_safety

for column_name, data_type in bronze_safety.dtypes:
    if data_type == "string":
        silver_safety = silver_safety.withColumn(
            column_name, F.trim(F.col(column_name))
        )

silver_safety = silver_safety.dropDuplicates(["incident_id"])

print("Safety incidents cleaned")
print("Rows:", silver_safety.count())

In [0]:
silver_table = "workspace.transportation_analytics.silver_safety_incidents"

if not spark.catalog.tableExists(silver_table):
    silver_safety.write.format("delta").saveAsTable(silver_table)
    print("Silver safety incidents table created")
    silver_safety.printSchema()
else:
    DeltaTable.forName(spark, silver_table).alias("t") \
        .merge(
            silver_safety.alias("s"),
            "t.incident_id = s.incident_id"
        ) \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()
    print("Silver safety incidents table updated using MERGE")

print("Rows:", spark.table(silver_table).count())

In [0]:
bronze_trailers = spark.table(
    "workspace.transportation_analytics.bronze_trailers"
)

print("Trailers loaded")
print("Rows:", bronze_trailers.count())

In [0]:
silver_trailers = bronze_trailers

for column_name, data_type in bronze_trailers.dtypes:
    if data_type == "string":
        silver_trailers = silver_trailers.withColumn(
            column_name, F.trim(F.col(column_name))
        )

silver_trailers = silver_trailers.dropDuplicates(["trailer_id"])

print("Trailers cleaned")
print("Rows:", silver_trailers.count())

In [0]:
silver_table = "workspace.transportation_analytics.silver_trailers"

if not spark.catalog.tableExists(silver_table):
    silver_trailers.write.format("delta").saveAsTable(silver_table)
    print("Silver trailers table created")
    silver_trailers.printSchema()
else:
    DeltaTable.forName(spark, silver_table).alias("t") \
        .merge(
            silver_trailers.alias("s"),
            "t.trailer_id = s.trailer_id"
        ) \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()
    print("Silver trailers table updated using MERGE")

print("Rows:", spark.table(silver_table).count())

In [0]:
bronze_trips = spark.table(
    "workspace.transportation_analytics.bronze_trips"
)

print("Trips loaded")
print("Rows:", bronze_trips.count())

In [0]:
silver_trips = bronze_trips

for column_name, data_type in bronze_trips.dtypes:
    if data_type == "string":
        silver_trips = silver_trips.withColumn(
            column_name, F.trim(F.col(column_name))
        )

silver_trips = silver_trips.dropDuplicates(["trip_id"])

print("Trips cleaned")
print("Rows:", silver_trips.count())

In [0]:
silver_table = "workspace.transportation_analytics.silver_trips"

if not spark.catalog.tableExists(silver_table):
    silver_trips.write.format("delta").saveAsTable(silver_table)
    print("Silver trips table created")
    silver_trips.printSchema()
else:
    DeltaTable.forName(spark, silver_table).alias("t") \
        .merge(
            silver_trips.alias("s"),
            "t.trip_id = s.trip_id"
        ) \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()
    print("Silver trips table updated using MERGE")

print("Rows:", spark.table(silver_table).count())

In [0]:
bronze_trucks = spark.table(
    "workspace.transportation_analytics.bronze_trucks"
)

print("Trucks loaded")
print("Rows:", bronze_trucks.count())

In [0]:
silver_trucks = bronze_trucks

for column_name, data_type in bronze_trucks.dtypes:
    if data_type == "string":
        silver_trucks = silver_trucks.withColumn(
            column_name, F.trim(F.col(column_name))
        )

silver_trucks = silver_trucks.dropDuplicates(["truck_id"])

print("Trucks cleaned")
print("Rows:", silver_trucks.count())

In [0]:
silver_table = "workspace.transportation_analytics.silver_trucks"

if not spark.catalog.tableExists(silver_table):
    silver_trucks.write.format("delta").saveAsTable(silver_table)
    print("Silver trucks table created")
    silver_trucks.printSchema()
else:
    DeltaTable.forName(spark, silver_table).alias("t") \
        .merge(
            silver_trucks.alias("s"),
            "t.truck_id = s.truck_id"
        ) \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()
    print("Silver trucks table updated using MERGE")

print("Rows:", spark.table(silver_table).count())

In [0]:
bronze_truck_utilization = spark.table(
    "workspace.transportation_analytics.bronze_truck_utilization_metrics"
)

print("Truck utilization metrics loaded")
print("Rows:", bronze_truck_utilization.count())

In [0]:
silver_truck_utilization = bronze_truck_utilization

for column_name, data_type in bronze_truck_utilization.dtypes:
    if data_type == "string":
        silver_truck_utilization = silver_truck_utilization.withColumn(
            column_name, F.trim(F.col(column_name))
        )

silver_truck_utilization = silver_truck_utilization.dropDuplicates(
    ["truck_id", "month"]
)

print("Truck utilization metrics cleaned")
print("Rows:", silver_truck_utilization.count())

In [0]:
silver_table = "workspace.transportation_analytics.silver_truck_utilization_metrics"

if not spark.catalog.tableExists(silver_table):
    silver_truck_utilization.write.format("delta").saveAsTable(silver_table)
    print("Silver truck utilization table created")
    silver_truck_utilization.printSchema()
else:
    DeltaTable.forName(spark, silver_table).alias("t") \
        .merge(
            silver_truck_utilization.alias("s"),
            "t.truck_id = s.truck_id AND t.month = s.month"
        ) \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()
    print("Silver truck utilization table updated using MERGE")

print("Rows:", spark.table(silver_table).count())